# Baseline And Intermittent Models

Simple and intermittent-demand baselines are mandatory. LightGBM is only meaningful if it beats these defensible baselines.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Image, Markdown, display
except Exception:
    def display(value):
        print(value)
    def Markdown(text):
        return text
    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename
        def __repr__(self):
            return f"Image(filename={self.filename!r})"

ROOT = Path.cwd()
if ROOT.name != "v7_rm_pm_forecast_planning":
    ROOT = Path("Ai miroservices/modeling/v7_rm_pm_forecast_planning").resolve()
OUT = ROOT / "outputs"
PLOTS = OUT / "plots"
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

def load_csv(name, **kwargs):
    path = OUT / name
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}. Run PYTHONPATH=. python3 -m pipeline.run_all first.")
    return pd.read_csv(path, **kwargs)

def load_json(name):
    path = OUT / name
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}. Run PYTHONPATH=. python3 -m pipeline.run_all first.")
    return json.loads(path.read_text())

def show_plot(name):
    path = PLOTS / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"Plot not generated: `{path}`"))

In [2]:
baseline = load_csv('baseline_leaderboard.csv')
baseline

,model,rows,materials,WAPE,MAE,RMSE,Bias,under_forecast_rate
0,croston_sba,1728,288,0.258377,446.166762,2401.367369,0.031150,0.434606
1,moving_avg_6,1728,288,0.282855,488.434896,2752.240619,0.082323,0.395255
2,moving_avg_3,1728,288,0.289843,500.502894,2724.766406,0.065316,0.400463
3,seasonal_naive,1728,288,0.348246,601.353009,3386.662683,0.075307,0.374421


In [3]:
rows = load_csv("baseline_backtest_rows.csv", parse_dates=["month"])
rows.head(30)

,model,material_id,material_code,description,material_type,month,actual,prediction,abs_error,error
0,seasonal_naive,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,2025-08-01,150.0,107.000000,43.000000,-43.000000
1,moving_avg_3,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,2025-08-01,150.0,120.000000,30.000000,-30.000000
2,moving_avg_6,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,2025-08-01,150.0,122.666667,27.333333,-27.333333
3,croston_sba,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,2025-08-01,150.0,122.922094,27.077906,-27.077906
4,seasonal_naive,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,2025-09-01,103.0,208.000000,105.000000,105.000000
5,moving_avg_3,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,2025-09-01,103.0,137.333333,34.333333,34.333333
6,moving_avg_6,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,2025-09-01,103.0,124.500000,21.500000,21.500000
7,croston_sba,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,2025-09-01,103.0,124.879885,21.879885,21.879885
8,seasonal_naive,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,2025-10-01,196.0,135.000000,61.000000,-61.000000
9,moving_avg_3,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,2025-10-01,196.0,129.666667,66.333333,-66.333333


In [4]:
rows.groupby("model").agg(
    rows=("actual", "size"),
    materials=("material_id", "nunique"),
    actual_sum=("actual", "sum"),
    prediction_sum=("prediction", "sum"),
)

,rows,materials,actual_sum,prediction_sum
model,,,,
croston_sba,1728,288,2983921.0,3.076870e+06
moving_avg_3,1728,288,2983921.0,3.178820e+06
moving_avg_6,1728,288,2983921.0,3.229566e+06
seasonal_naive,1728,288,2983921.0,3.208631e+06


In [5]:
classes = load_csv("demand_classification.csv")
classes.groupby("fms_class").agg(
    materials=("material_id", "count"),
    avg_nonzero_rate=("nonzero_rate", "mean"),
    avg_cv=("cv", "mean"),
)

,materials,avg_nonzero_rate,avg_cv
fms_class,,,
F,235,0.986879,0.481699
M,15,0.659259,0.966236
S,38,0.070175,0.634522
